<a href="https://colab.research.google.com/github/SantuMartire/Practicas-de-Machine-Learning/blob/main/Preentrega_Proyeco_Integrador_Diabetes_Santiago_M%C3%A1rtire.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Integrador de Machine Learning
## Predicción de Diabetes en pacientes


**Alumno:** Santiago Mártire

---

# *Requerimientos:*



1.   Selección del dataset: indicar origen, breve descripción del problema que aborda y justificación de su elección.

1.   Análisis exploratorio inicial (EDA): distribución de variables, detección de valores faltantes, análisis de correlaciones, visualizaciones básicas.


1.   Limpieza de datos: tratamiento de valores nulos, duplicados, outliers (si corresponde).

2.   Transformaciones básicas: codificación de variables categóricas, escalado de variables numéricas, creación o eliminación de variables (si fuera necesario).



2.   Selección de variables relevantes para el modelo.



2.   División del dataset en conjuntos de entrenamiento y prueba.


---


## 1. 📦 **Selección del Dataset**

### Origen del Dataset

El dataset utilizado es el **Diabetes prediction dataset** publicado por **Mohammed Mustafa** en Kaggle:

🔗 kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset



---

### Descripción del Problema

Se busca construir un modelo de clasificación binaria capaz de predecir si un paciente presenta diabetes, a partir de variables clínicas y de estilo de vida, con el fin de asistir en el diagnóstico temprano


### Justificación de la elección



---

## 2. 🔍 **Análisis Exploratorio de Datos** (EDA)

El Análisis Exploratorio de Datos (EDA) es el primer paso para entender la estructura y el contenido del dataset antes de cualquier procesamiento. Nos permite identificar:

- La forma general del dataset (dimensiones, tipos de datos)
- La distribución de las variables
- La presencia de valores nulos o anómalos
- Las correlaciones entre variables

---

### Vista General del Dataset

In [1]:
# Librerías para el manejo de datos
import pandas as pd
import numpy as np

# Librerías para creación de gráficos
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Librerías para pruebas estadísticas
from scipy import stats as st
import math as mt

# Libreria para importar directamente desde Kaggle
import kagglehub

# Extra: acceso a comandos del Sistema Operativo
import os

In [2]:
# ==========================================================
# CARGAMOS EL DATASET
# ==========================================================

# Si el archivo ya está disponible en una ruta local, lo usamos directamente.
# En caso contrario, lo descargamos desde Kaggle.

ruta_local = "/content/diabetes_prediction_dataset.csv"

if os.path.exists(ruta_local):
    ruta_csv = ruta_local
else:
    import kagglehub
    ruta_dataset = kagglehub.dataset_download("iammustafatz/diabetes-prediction-dataset")
    ruta_csv = os.path.join(ruta_dataset, "diabetes_prediction_dataset.csv")

df = pd.read_csv(ruta_csv)

print("Dataset cargado correctamente.")
print("Dimensiones:", df.shape)

display(df.head())

Using Colab cache for faster access to the 'diabetes-prediction-dataset' dataset.
Dataset cargado correctamente.
Dimensiones: (100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [3]:
# ==========================================================
# EXPLORACIÓN INICIAL DE LA ESTRUCTURA DEL DATASET
# ==========================================================
print("Dimensiones del dataset:", df.shape)

Dimensiones del dataset: (100000, 9)


In [4]:
print("\nInformación general (tpos de datos y valores no nulos):")
df.info()


Información general (tpos de datos y valores no nulos):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   gender               100000 non-null  object 
 1   age                  100000 non-null  float64
 2   hypertension         100000 non-null  int64  
 3   heart_disease        100000 non-null  int64  
 4   smoking_history      100000 non-null  object 
 5   bmi                  100000 non-null  float64
 6   HbA1c_level          100000 non-null  float64
 7   blood_glucose_level  100000 non-null  int64  
 8   diabetes             100000 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.9+ MB


In [6]:
print("Columnas del dataset:")
print(df.columns.tolist())

Columnas del dataset:
['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history', 'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes']


In [7]:
print("\nResumen estadístico de las variables numéricas:")
display(df.describe())


Resumen estadístico de las variables numéricas:


,age,hypertension,heart_disease,bmi,HbA1c_level,blood_glucose_level,diabetes
count,100000.000000,100000.00000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,41.885856,0.07485,0.039420,27.320767,5.527507,138.058060,0.085000
std,22.516840,0.26315,0.194593,6.636783,1.070672,40.708136,0.278883
min,0.080000,0.00000,0.000000,10.010000,3.500000,80.000000,0.000000
25%,24.000000,0.00000,0.000000,23.630000,4.800000,100.000000,0.000000
50%,43.000000,0.00000,0.000000,27.320000,5.800000,140.000000,0.000000
75%,60.000000,0.00000,0.000000,29.580000,6.200000,159.000000,0.000000
max,80.000000,1.00000,1.000000,95.690000,9.000000,300.000000,1.000000


In [8]:
# ==========================================================
# REVISAMOS TIPOS DE DATOS Y VALORES FALTANTES
# ==========================================================

print("\nCantidad de valores nulos por columna:\n")
print(df.isnull().sum())
print("\n Porcentaje (%):")
print((df.isnull().sum() / len(df) * 100).round(2))



Cantidad de valores nulos por columna:

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
diabetes               0
dtype: int64

 Porcentaje (%):
gender                 0.0
age                    0.0
hypertension           0.0
heart_disease          0.0
smoking_history        0.0
bmi                    0.0
HbA1c_level            0.0
blood_glucose_level    0.0
diabetes               0.0
dtype: float64
